In [1]:
df = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/ml_ready_dataset/"
)

In [2]:
from pyspark.sql.functions import col, date_trunc, avg, sum

import pandas as pd
import numpy as np

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [3]:
%pip install prophet

In [4]:
from pyspark.sql.functions import col, date_trunc, avg, sum

import pandas as pd
import numpy as np

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [5]:
df.printSchema()

df.select(
    "tstp",
    "energy_kwh",
    "rolling_mean_24h",
    "weekly_avg_energy",
    "season_avg_energy"
).show(5, False)

In [6]:
df = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/ml_ready_dataset/"
)

In [7]:
df.printSchema()

df.select(
    "tstp",
    "energy_kwh",
    "rolling_mean_24h",
    "weekly_avg_energy",
    "season_avg_energy"
).show(5, False)

In [8]:
prophet_spark_df = df.withColumn(
    "ds",
    date_trunc("hour", col("tstp"))
).groupBy("ds").agg(
    avg("energy_kwh").alias("y")
).orderBy("ds")

In [9]:
prophet_spark_df = prophet_spark_df.dropna()

In [10]:
prophet_spark_df.show(10, False)

In [11]:
prophet_pdf = prophet_spark_df.toPandas()

In [12]:
prophet_pdf.isnull().sum()

In [13]:
prophet_pdf["ds"] = pd.to_datetime(prophet_pdf["ds"])
prophet_pdf = prophet_pdf.sort_values("ds")

In [14]:
train = prophet_pdf[prophet_pdf["ds"] < "2014-01-01"]
test = prophet_pdf[prophet_pdf["ds"] >= "2014-01-01"]

In [15]:
model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True
)

model.fit(train)

In [16]:
future = model.make_future_dataframe(
    periods=len(test),
    freq="H"
)

In [17]:
forecast = model.predict(future)

In [18]:
pred = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]

result = test.merge(
    pred,
    on="ds",
    how="left"
)

result.head()

In [19]:
mae = mean_absolute_error(result["y"], result["yhat"])
rmse = np.sqrt(mean_squared_error(result["y"], result["yhat"]))

print("MAE :", mae)
print("RMSE :", rmse)

In [20]:
model.plot(forecast)

In [21]:
model.plot_components(forecast)

In [22]:
result_spark = spark.createDataFrame(result)

result_spark.write.format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://curated@energybigdatastorage.dfs.core.windows.net/prophet_predictions/"
    )

In [23]:
test_predictions = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/prophet_predictions/"
)

test_predictions.show(5, False)